##Passo 1
Aqui iremos fazer a organização do ambiente. Vou utilizar de base a maneira como aprendemos na aula prática, visando manter um padrão que facilite a organização e também o entendimento do passo a passo.



Vou criar primeiro os parâmetros, a fim de facilitar a execução

In [0]:

catalog = "catalogo_de_filmes"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

##Passo 2:
Criar os schemas, o catálogo e o volume

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

print("Catalog, schemas e volume prontos.")

##Passo 3: Upload e validação do upload.
Arquivos esperados:

 - `credits_and_tags_IMDB_TMDB.csv`
 - `movies_financials_IMDB_TMDB.csv`
 - `movies_info_TMDB_IMDB.csv`
 - `movies_metrics_IMDB_TMDB.csv`
 - `movies_reviews.csv`

In [0]:
expected_files = [
    "credits_and_tags_IMDB_TMDB.csv",
"movies_financials_IMDB_TMDB.csv",
"movies_info_TMDB_IMDB.csv",
"movies_metrics_IMDB_TMDB.csv",
"movies_reviews.csv"
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar '{landing_path}'. Faça o upload dos arquivos antes de continuar.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
else:
    print("[OK] Todos os arquivos esperados estão na landing zone:")
    for f in dbutils.fs.ls(landing_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")